# Constrained PXP Chain

This notebook replaces the old `ConstrainedPXP.jl` script with a more explanatory walkthrough. The PXP model is a good example of why `ProjectedBasis` is useful: the interesting Hilbert space is not the full tensor-product space, but the subspace with no adjacent excitations.

In [ ]:
import Pkg

function find_repo_root(start = pwd())
    dir = abspath(start)
    while true
        if isfile(joinpath(dir, "Project.toml")) && isfile(joinpath(dir, "src", "EDKit.jl"))
            return dir
        end
        parent = dirname(dir)
        parent == dir && error("Could not locate EDKit.jl repo root from $(start)")
        dir = parent
    end
end

const DEV = true
const REPO_ROOT = find_repo_root()
Pkg.activate(REPO_ROOT)
if DEV
    pushfirst!(LOAD_PATH, REPO_ROOT)
end

using EDKit
using LinearAlgebra
using Statistics


## Constraint and projected basis

For open boundaries, the Rydberg blockade constraint is simply that no two neighboring sites may both be excited. We implement that as a boolean predicate and use it to build a `ProjectedBasis`.

In [ ]:
function pxp_obc_constraint(v::AbstractVector{<:Integer})
    all(v[i] == 0 || v[i + 1] == 0 for i in 1:length(v)-1)
end

L = 10
basis = ProjectedBasis(L = L, f = pxp_obc_constraint)
size(basis, 1)


## Build the constrained Hamiltonian

The local PXP term flips the middle spin only when both neighbors are in the allowed state. In the projected basis we can build the Hamiltonian directly and diagonalize it exactly at this small size.

In [ ]:
local_term = let projector = Diagonal([1, 1, 1, 0, 1, 1, 0, 0])
    projector * kron(I(2), spin("X"), I(2)) * projector
end

inds = [[i, i + 1, i + 2] for i in 1:L-2]
H = operator(fill(local_term, L - 2), inds, basis)
vals, vecs = eigen(Hermitian(H))

neel_state = [isodd(i) ? 1 : 0 for i in 1:L]
psi0 = productstate(neel_state, basis)
overlaps = abs2.(vecs' * psi0)
scar_index = argmax(overlaps)

summary = (
    constrained_dimension = size(basis, 1),
    ground_energy = vals[1],
    strongest_neel_overlap = overlaps[scar_index],
    scar_energy = vals[scar_index],
    scar_entropy = ent_S(vecs[:, scar_index], 1:L÷2, basis),
)
summary


## Plot the Neel-state overlap across the spectrum

A standard first diagnostic in small PXP calculations is to look for unusually large overlap with the Neel state. Those atypical eigenstates are the ones usually associated with the scar story.

In [ ]:
if isdefined(Main, :IJulia)
    using Plots
    scatter(vals, overlaps; xlabel = "energy", ylabel = "|<E|Neel>|^2", label = nothing, title = "Neel overlap across the spectrum")
else
    @info "Skipping Plots outside IJulia; returning spectrum and overlaps instead"
    (; energies = vals, overlaps = overlaps)
end


## Takeaway

The important package idea here is not just the PXP model itself. It is that `ProjectedBasis` lets you formulate the Hamiltonian directly in the physically relevant constrained space. That keeps the example small and makes exact diagonalization practical.